# Pythia Family -- Pruning as a Phase Transition

This notebook applies the **critiPrune** framework to the [Pythia](https://github.com/EleutherAI/pythia) transformer model family (14M to 6.9B parameters), testing whether structured MLP pruning exhibits a universal sigmoid phase transition and power-law scaling.

---

## Pipeline

1. **Calibration**: Collect MLP activation statistics (input norms + post-GELU intermediate norms) over 128 samples from the C4 dataset.

2. **WANDA scoring**: Compute per-neuron importance as $|W_{\text{up}}| \times \|x\|$, combining weight magnitude with activation statistics ([Sun et al., 2023](https://arxiv.org/abs/2306.11695)).
3. **K-sweep**: For each fraction $K$ of $d_{\text{ff}}$ neurons kept (1% to 100%), apply structured top-K pruning via forward hooks on the MLP gate projection, then measure perplexity on WikiText-2.
4. **Sigmoid fit**: Fit the recovery curve to $A(K) = A_0 + (A_\infty - A_0) / (1 + e^{-\beta(K - K_0)})$ and extract the critical threshold $K_0$ and steepness $\beta$.
5. **Scaling laws**: Fit joint power laws $K_0 = c \cdot d_{\text{ff}}^\alpha \cdot L^\gamma$ and $\beta = c' \cdot d_{\text{ff}}^{\alpha'} \cdot L^{\gamma'}$ across the model family.

## Notebook Structure

| Section | Contents |
|---------|----------|
| **Init** | Imports, constants, HuggingFace authentication |
| **Pruning** | `MLPActivationCollector`, WANDA scoring, `WandaTopKPruner` hooks, perplexity evaluation, sigmoid fitting |
| **Experiments** | `run_single_model()` -- full pipeline for one Pythia checkpoint |
| **Scaling Law Fits** | Power-law regression of $(K_0, \beta)$ vs $(d_{\text{ff}}, L)$ with adjusted $R^2$ |
| **Visualization** | Sigmoid recovery curves per model, scaling law scatter plots |
| **Main** | CLI entry point with `--models`, `--device`, and output directory arguments |

## Requirements

- **GPU**: Colab T4 handles models up to 2.8B; A100 required for 6.9B
- **Dependencies**: `torch`, `transformers`, `datasets`, `accelerate`, `scipy`, `numpy`, `matplotlib`
- **HuggingFace token**: Required for model downloads (set in the Init cell)

## Init

In [ ]:
"""
Effective Coupling Constants Across the Pythia Model Family
============================================================
Tests whether the sigmoid pruning law and scaling relations discovered
on FC networks (MNIST) hold for transformer LLMs.

Run on Google Colab with GPU runtime (T4 sufficient for 70M-2.8B).

Usage
-----
  !pip install transformers datasets accelerate scipy matplotlib -q
  !python pythia_scaling.py                     # default: 4 small models
  !python pythia_scaling.py --models all        # full suite (needs A100 for 6.9B)
  !python pythia_scaling.py --models 70m 410m   # custom selection

Method
------
For each Pythia model we:
  1. Collect MLP activation statistics over 128 calibration samples (C4).
  2. Compute per-neuron WANDA importance scores (|W|·‖x‖ aggregated).
  3. Sweep K = fraction of neurons kept from 1% to 100%, measuring
     perplexity recovery via forward hooks that zero pruned neurons.
  4. Fit the sigmoid: recovery(K) = A₀ + (A∞-A₀)/(1+exp(−β(K-K₀)))
  5. Extract (K₀, β, g_eff=e^{-β}) per model.
  6. Fit joint scaling laws across the model family.

Output
------
  pythia_figures/
    pythia_recovery_curves.png      — sigmoid fits per model
    pythia_scaling_laws.png         — K₀, β, g vs architecture
    pythia_parameter_heatmap.png    — summary table as heatmap
    pythia_results.json             — all numerical results
"""

import os, sys, json, time, gc, argparse, warnings
warnings.filterwarnings('ignore')

import numpy as np
from scipy.optimize import curve_fit

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ========================
#  CONFIGURATION
# ========================

OUTPUT_DIR = './pythia_figures'

# Pythia model family: (name_suffix, L, d_model, d_ff)
# d_ff = 4 * d_model for all Pythia models
PYTHIA_MODELS = {
    '14m':  dict(hf='EleutherAI/pythia-14m',   L=6,  H=128,   d_ff=512),
    '31m':  dict(hf='EleutherAI/pythia-31m',   L=6,  H=256,   d_ff=1024),
    '70m':  dict(hf='EleutherAI/pythia-70m',   L=6,  H=512,   d_ff=2048),
    '160m': dict(hf='EleutherAI/pythia-160m',  L=12, H=768,   d_ff=3072),
    '410m': dict(hf='EleutherAI/pythia-410m',  L=24, H=1024,  d_ff=4096),
    '1b':   dict(hf='EleutherAI/pythia-1b',    L=16, H=2048,  d_ff=8192),
    '1.4b': dict(hf='EleutherAI/pythia-1.4b',  L=24, H=2048,  d_ff=8192),
    '2.8b': dict(hf='EleutherAI/pythia-2.8b',  L=32, H=2560,  d_ff=10240),
    '6.9b': dict(hf='EleutherAI/pythia-6.9b',  L=32, H=4096,  d_ff=16384),
}

# Default models (fit on T4 16GB comfortably)
DEFAULT_MODELS = ['14m', '31m', '70m', '160m', '410m', '1b', '1.4b']
# DEFAULT_MODELS = ['2.8b']

# Sparsity sweep: fraction of d_ff neurons KEPT
# Dense at both ends, sparser in the middle for efficiency
K_FRACTIONS = np.unique(np.sort(np.concatenate([
    np.arange(0.01, 0.06, 0.01),       # 1%–5%  (fine resolution at low K)
    np.arange(0.05, 0.20, 0.025),       # 5%–20%
    np.arange(0.20, 0.55, 0.05),        # 20%–50%
    np.arange(0.50, 1.01, 0.10),        # 50%–100%
])))

# Calibration
N_CALIB_SAMPLES = 128
CALIB_SEQ_LEN   = 128

# Perplexity evaluation
EVAL_DATASET    = 'wikitext'          # 'wikitext' or 'pile-10k'
EVAL_MAX_TOKENS = 40_000             # cap for speed; increase for precision
EVAL_STRIDE     = 1024
EVAL_MAX_LENGTH = 2048


## Pruning

### Activation Collector

In [ ]:
class MLPActivationCollector:
    """Collect squared activation norms at MLP up-projection inputs
    and post-GELU intermediate activations."""

    def __init__(self, model):
        self.model = model
        self.n_layers = model.config.num_hidden_layers
        self.hooks = []
        self.input_sq = {}     # layer_idx -> [d_model]
        self.inter_sq = {}     # layer_idx -> [d_ff]
        self.n_tokens = 0

    def _hook_input(self, idx):
        def fn(module, inp, out):
            x = inp[0].detach().float().reshape(-1, inp[0].shape[-1])
            if idx not in self.input_sq:
                self.input_sq[idx] = torch.zeros(x.shape[1], device=x.device)
            self.input_sq[idx] += (x ** 2).sum(dim=0)
            if idx == 0:
                self.n_tokens += x.shape[0]
        return fn

    def _hook_inter(self, idx):
        def fn(module, inp, out):
            # inp to dense_4h_to_h = post-GELU activations
            x = inp[0].detach().float().reshape(-1, inp[0].shape[-1])
            if idx not in self.inter_sq:
                self.inter_sq[idx] = torch.zeros(x.shape[1], device=x.device)
            self.inter_sq[idx] += (x ** 2).sum(dim=0)
        return fn

    def register(self):
        for i, layer in enumerate(self.model.gpt_neox.layers):
            self.hooks.append(
                layer.mlp.dense_h_to_4h.register_forward_hook(self._hook_input(i)))
            self.hooks.append(
                layer.mlp.dense_4h_to_h.register_forward_hook(self._hook_inter(i)))

    def remove(self):
        for h in self.hooks:
            h.remove()
        self.hooks.clear()

    def get_norms(self):
        """Return RMS norms (not raw squared sums)."""
        input_norms, inter_norms = {}, {}
        for i in range(self.n_layers):
            input_norms[i] = torch.sqrt(self.input_sq[i] / self.n_tokens)
            inter_norms[i] = torch.sqrt(self.inter_sq[i] / self.n_tokens)
        return input_norms, inter_norms

### Wanda Scoring

In [ ]:
def compute_wanda_scores(model, input_norms, inter_norms):
    """
    Structured WANDA: per-neuron importance = sum of |W_up[j,:]| * input_norm[:]
    combined with down-projection column norm * intermediate activation norm.
    Returns dict {layer_idx: Tensor[d_ff]}.
    """
    scores = {}
    for i, layer in enumerate(model.gpt_neox.layers):
        W_up = layer.mlp.dense_h_to_4h.weight.data.float()    # [d_ff, d_model]
        W_down = layer.mlp.dense_4h_to_h.weight.data.float()  # [d_model, d_ff]

        # Up-projection WANDA: how important is neuron j based on input signal?
        s_up = (W_up.abs() * input_norms[i].unsqueeze(0)).sum(dim=1)  # [d_ff]

        # Down-projection × activation: how much does neuron j contribute to output?
        s_down = W_down.norm(p=2, dim=0) * inter_norms[i]  # [d_ff]

        # Combine (normalize each to [0,1] then sum)
        s_up_n = s_up / (s_up.max() + 1e-12)
        s_down_n = s_down / (s_down.max() + 1e-12)
        scores[i] = s_up_n + s_down_n

    return scores

### Top-K via Hooks

In [ ]:
class WandaTopKPruner:
    """
    Apply structured neuron pruning using precomputed WANDA scores.
    For each layer, only the top-K neurons (by WANDA score) pass signal;
    the rest are zeroed after GELU.
    """

    def __init__(self, model, wanda_scores, keep_fraction=1.0):
        self.model = model
        self.wanda_scores = wanda_scores
        self.d_ff = model.config.intermediate_size
        self.keep_fraction = keep_fraction
        self._handles = []

        # Precompute masks for each layer
        K = max(1, int(self.d_ff * keep_fraction))
        self.masks = {}
        for i in range(model.config.num_hidden_layers):
            s = wanda_scores[i]
            _, top_idx = torch.topk(s, K)
            mask = torch.zeros(self.d_ff, device=s.device, dtype=torch.float16)
            mask[top_idx] = 1.0
            self.masks[i] = mask  # [d_ff]

    def _make_hook(self, layer_idx):
        mask = self.masks[layer_idx]
        def hook_fn(module, input, output):
            return output * mask.unsqueeze(0).unsqueeze(0)
        return hook_fn

    def enable(self):
        for i, layer in enumerate(self.model.gpt_neox.layers):
            h = layer.mlp.act.register_forward_hook(self._make_hook(i))
            self._handles.append(h)

    def disable(self):
        for h in self._handles:
            h.remove()
        self._handles.clear()

    def __enter__(self):
        self.enable()
        return self

    def __exit__(self, *args):
        self.disable()

### Perplexity Evaluation

In [ ]:
def evaluate_perplexity(model, tokenizer, dataset='wikitext',
                        max_tokens=40_000, stride=1024,
                        max_length=2048, device='cuda'):
    """Sliding-window perplexity on WikiText-2 or Pile-10k subset."""
    from datasets import load_dataset

    if dataset == 'wikitext':
        test = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
        text = '\n\n'.join(test['text'])
    else:
        ds = load_dataset('NeelNanda/pile-10k', split='train')
        text = '\n\n'.join(ds['text'][:300])

    encodings = tokenizer(text, return_tensors='pt')
    seq_len = min(encodings.input_ids.size(1), max_tokens)

    nll_sum = 0.0
    n_tokens = 0
    prev_end = 0

    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = encodings.input_ids[:, begin:end].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            loss = model(input_ids, labels=target_ids).loss

        num_scored = (target_ids != -100).sum().item() - 1
        if num_scored > 0:
            nll_sum += loss.float().item() * num_scored
            n_tokens += num_scored

        prev_end = end
        if end >= seq_len:
            break

    if n_tokens == 0:
        return float('inf')
    ppl = np.exp(nll_sum / n_tokens)
    return ppl

### Sigmoid Fit

In [ ]:
def sigmoid_fn(K, A_inf, A_0, K_0, beta):
    K = np.asarray(K, dtype=float)
    return A_0 + (A_inf - A_0) / (1.0 + np.exp(
        np.clip(-beta * (K - K_0), -500, 500)))


def fit_sigmoid(k_fracs, recoveries):
    """
    Fit recovery(K) = A₀ + (A∞−A₀)/(1+exp(−β(K−K₀)))
    where K is the fraction of neurons kept (0 to 1).

    Returns (popt, R²) or (None, None).
    """
    k_arr = np.array(k_fracs)
    r_arr = np.array(recoveries)

    try:
        p0 = [max(r_arr), min(r_arr), np.median(k_arr), 10.0]
        bounds = (
            [0.0, -0.1, 0.0, 0.1],
            [1.5, 1.0, 1.0, 200.0],
        )
        popt, pcov = curve_fit(sigmoid_fn, k_arr, r_arr, p0=p0,
                                bounds=bounds, maxfev=30000)
        resid = r_arr - sigmoid_fn(k_arr, *popt)
        ss_res = np.sum(resid ** 2)
        ss_tot = np.sum((r_arr - r_arr.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        return popt, r2
    except Exception as e:
        print(f"    Sigmoid fit failed: {e}")
        return None, None



## Experiments

### Single Model Experiment

In [ ]:
def run_single_model(model_key, device='cuda',
                     eval_max_tokens=EVAL_MAX_TOKENS,
                     n_calib_samples=N_CALIB_SAMPLES):
    """
    Full pipeline for one Pythia model:
    load → calibrate → score → sweep K → fit sigmoid → clean up.
    """
    spec = PYTHIA_MODELS[model_key]
    hf_name = spec['hf']
    d_ff = spec['d_ff']

    print(f"\n{'═'*60}")
    print(f"  {hf_name}  (L={spec['L']}, H={spec['H']}, d_ff={d_ff})")
    print(f"{'═'*60}")

    # --- Load model --------------------------------------
    t0 = time.time()
    print(f"  Loading model ...", end='', flush=True)
    token = input("Enter HuggingFace token:")
    model = AutoModelForCausalLM.from_pretrained(
        hf_name, torch_dtype=torch.float16, token=token,
        low_cpu_mem_usage=True).to(device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(hf_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"  [{time.time()-t0:.0f}s]")

    # --- Baseline perplexity --------------------------------
    print(f"  Baseline perplexity ...", end='', flush=True)
    t1 = time.time()
    ppl_baseline = evaluate_perplexity(
        model, tokenizer, dataset=EVAL_DATASET,
        max_tokens=eval_max_tokens, stride=EVAL_STRIDE,
        max_length=EVAL_MAX_LENGTH, device=device)
    print(f"  {ppl_baseline:.2f}  [{time.time()-t1:.0f}s]")

    # -- Calibration: collect activation norms -------------------
    print(f"  Collecting activations ({n_calib_samples} samples) ...", end='', flush=True)
    t1 = time.time()
    from datasets import load_dataset
    c4 = load_dataset('allenai/c4', 'en', split='train', streaming=True,
                       trust_remote_code=True)
    cal_texts = []
    for sample in c4:
        cal_texts.append(sample['text'])
        if len(cal_texts) >= n_calib_samples:
            break

    collector = MLPActivationCollector(model)
    collector.register()
    with torch.no_grad():
        for text in cal_texts:
            ids = tokenizer(text, return_tensors='pt', truncation=True,
                            max_length=CALIB_SEQ_LEN).to(device)
            model(**ids)
    collector.remove()
    input_norms, inter_norms = collector.get_norms()
    del collector
    print(f"  [{time.time()-t1:.0f}s]")

    # -- WANDA neuron scores ----------------------------
    print(f"  Computing WANDA scores ...", end='', flush=True)
    t1 = time.time()
    wanda_scores = compute_wanda_scores(model, input_norms, inter_norms)
    del input_norms, inter_norms
    print(f"  [{time.time()-t1:.0f}s]")

    # -- Sweep K fractions ------------------------------------
    print(f"  Sweeping {len(K_FRACTIONS)} sparsity levels ...")
    k_fracs = []
    ppls = []
    recoveries = []

    for kf in K_FRACTIONS:
        t1 = time.time()
        pruner = WandaTopKPruner(model, wanda_scores, keep_fraction=kf)
        pruner.enable()

        ppl = evaluate_perplexity(
            model, tokenizer, dataset=EVAL_DATASET,
            max_tokens=eval_max_tokens, stride=EVAL_STRIDE,
            max_length=EVAL_MAX_LENGTH, device=device)

        pruner.disable()

        # Recovery metric: 1 / (1 + L_sparse / L_baseline)
        # This maps [baseline_ppl, ∞] → [0.5, 0] and is 1/(1+1)=0.5 at baseline
        # Better: inverse relative loss
        # recovery = baseline_loss / sparse_loss ∈ (0, 1]
        loss_baseline = np.log(ppl_baseline)
        loss_sparse = np.log(max(ppl, 1.01))
        recovery = min(loss_baseline / loss_sparse, 1.0)

        k_fracs.append(float(kf))
        ppls.append(float(ppl))
        recoveries.append(float(recovery))

        K_abs = int(d_ff * kf)
        print(f"    K={kf:5.1%} ({K_abs:>5}/{d_ff})  "
              f"ppl={ppl:>10.2f}  recovery={recovery:.4f}  [{time.time()-t1:.0f}s]")

    # -- Fit sigmoid -----------------------------------
    popt, r2 = fit_sigmoid(k_fracs, recoveries)

    result = {
        'model': model_key,
        'hf_name': hf_name,
        'L': spec['L'],
        'H': spec['H'],
        'd_ff': d_ff,
        'n_params_M': sum(p.numel() for p in model.parameters()) / 1e6,
        'ppl_baseline': float(ppl_baseline),
        'k_fracs': k_fracs,
        'ppls': ppls,
        'recoveries': recoveries,
    }

    if popt is not None:
        A_inf, A_0, K_0, beta = popt
        g_eff = np.exp(-beta)
        result.update({
            'sigmoid_A_inf': float(A_inf),
            'sigmoid_A_0': float(A_0),
            'sigmoid_K_0': float(K_0),
            'sigmoid_K_0_abs': int(K_0 * d_ff),
            'sigmoid_beta': float(beta),
            'sigmoid_g_eff': float(g_eff),
            'sigmoid_R2': float(r2),
        })
        print(f"\n  Sigmoid fit: K₀={K_0:.3f} ({int(K_0*d_ff)}/{d_ff})  "
              f"β={beta:.2f}  g={g_eff:.4f}  R²={r2:.4f}")
    else:
        result['sigmoid_R2'] = None
        print(f"\n  Sigmoid fit FAILED")

    # -- Cleanup ------------
    del model, tokenizer, wanda_scores
    gc.collect()
    torch.cuda.empty_cache()

    return result



### Scaling Law fits

In [ ]:
def power_law_2d(HL, a, alpha, gamma):
    H, L = HL
    return a * np.power(H, alpha) * np.power(L, gamma)


def fit_scaling_laws(results):
    good = [r for r in results
            if r.get('sigmoid_R2') is not None and r['sigmoid_R2'] > 0.80]
    if len(good) < 3:
        print("  Too few good fits for scaling law analysis")
        return None

    H = np.array([r['d_ff'] for r in good], dtype=float)   # use d_ff as "width"
    L = np.array([r['L'] for r in good], dtype=float)
    K0 = np.array([r['sigmoid_K_0'] for r in good])        # fractional
    K0_abs = np.array([r['sigmoid_K_0_abs'] for r in good], dtype=float)
    beta = np.array([r['sigmoid_beta'] for r in good])
    g = np.array([r['sigmoid_g_eff'] for r in good])

    scaling = {}
    print(f"\n{'═'*60}")
    print(f"  SCALING LAW ANALYSIS ({len(good)} models)")
    print(f"{'═'*60}")

    for name, arr, p0, bnd in [
        ('K0_abs', K0_abs, [0.1, 0.5, 0.5], ([0, -2, -2], [1e6, 3, 3])),
        ('beta', beta, [100, -0.5, -0.5], ([0, -3, -3], [1e6, 3, 3])),
        ('g_eff', g, [0.5, 0.1, 0.1], ([0, -3, -3], [2, 3, 3])),
    ]:
        try:
            popt, pcov = curve_fit(power_law_2d, (H, L), arr,
                                    p0=p0, bounds=bnd, maxfev=10000)
            pred = power_law_2d((H, L), *popt)
            ss_res = np.sum((arr - pred)**2)
            ss_tot = np.sum((arr - arr.mean())**2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
            perr = np.sqrt(np.diag(pcov))
            a, al, ga = popt
            sym = {'K0_abs': 'K₀', 'beta': 'β', 'g_eff': 'g_eff'}[name]
            scaling[name] = dict(a=float(a), alpha=float(al), gamma=float(ga),
                                  R2=float(r2))
            print(f"\n  {sym} = {a:.4f} × d_ff^{al:.3f} × L^{ga:.3f}")
            print(f"       ± ({perr[0]:.3f}, {perr[1]:.3f}, {perr[2]:.3f})")
            print(f"       R² = {r2:.4f}")
        except Exception as e:
            print(f"  {name} fit failed: {e}")

    # K0_frac statistics
    print(f"\n  K₀ (fractional) statistics:")
    print(f"    mean = {K0.mean():.3f} ± {K0.std():.3f}")
    print(f"    range = [{K0.min():.3f}, {K0.max():.3f}]")
    print(f"{'═'*60}")

    return scaling

### Visualization

In [ ]:

def make_plots(results, scaling, output_dir):
    good = [r for r in results
            if r.get('sigmoid_R2') is not None and r['sigmoid_R2'] > 0.80]
    if len(good) < 2:
        print("  Not enough data for plots"); return []

    os.makedirs(output_dir, exist_ok=True)
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(good)))
    paths = []

    # ---------------------------------------------------
    #  Figure 1: Recovery curves + sigmoid fits
    # ----------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    ax = axes[0]
    for i, r in enumerate(sorted(good, key=lambda x: x['n_params_M'])):
        kf = np.array(r['k_fracs'])
        rec = np.array(r['recoveries'])
        ax.scatter(kf * 100, rec, s=25, color=colors[i], alpha=0.8, zorder=5)

        if r.get('sigmoid_R2') and r['sigmoid_R2'] > 0.80:
            kf_fine = np.linspace(0.01, 1.0, 300)
            fit = sigmoid_fn(kf_fine, r['sigmoid_A_inf'], r['sigmoid_A_0'],
                              r['sigmoid_K_0'], r['sigmoid_beta'])
            ax.plot(kf_fine * 100, fit, color=colors[i], lw=2,
                    label=f'{r["model"]} (g={r["sigmoid_g_eff"]:.3f}, '
                          f'K₀={r["sigmoid_K_0"]*100:.1f}%, '
                          f'R²={r["sigmoid_R2"]:.3f})')

    ax.set_xlabel('MLP neurons kept (%)', fontsize=12)
    ax.set_ylabel('Loss recovery (baseline_loss / sparse_loss)', fontsize=11)
    ax.set_title('WANDA Pruning Recovery Curves — Pythia Family', fontsize=12)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 105)

    # Bar chart of parameters
    ax = axes[1]
    models = [r['model'] for r in sorted(good, key=lambda x: x['n_params_M'])]
    K0_vals = [r['sigmoid_K_0'] * 100 for r in sorted(good, key=lambda x: x['n_params_M'])]
    g_vals = [r['sigmoid_g_eff'] for r in sorted(good, key=lambda x: x['n_params_M'])]
    x = np.arange(len(models))
    w = 0.35
    bars1 = ax.bar(x - w/2, K0_vals, w, color=[colors[i] for i in range(len(models))],
                    alpha=0.85, edgecolor='black', lw=0.6, label='$K_0$ (%)')
    ax.set_ylabel('$K_0$ (% of MLP neurons)', fontsize=11)

    ax2 = ax.twinx()
    bars2 = ax2.bar(x + w/2, g_vals, w, color=[colors[i] for i in range(len(models))],
                     alpha=0.45, edgecolor='black', lw=0.6, hatch='//', label='$g_{eff}$')
    ax2.set_ylabel('$g_{eff} = e^{-\\beta}$', fontsize=11, color='dimgray')

    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=9, rotation=30, ha='right')
    ax.set_title('Sigmoid Parameters per Model', fontsize=12)
    ax.grid(axis='y', alpha=0.3)

    lines = [plt.Rectangle((0,0),1,1, color='gray', alpha=0.85),
             plt.Rectangle((0,0),1,1, color='gray', alpha=0.45, hatch='//')]
    ax.legend(lines, ['$K_0$ (%)', '$g_{eff}$'], fontsize=9, loc='upper left')

    plt.tight_layout()
    p = os.path.join(output_dir, 'pythia_recovery_curves.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    paths.append(p); print(f"  Saved: {p}")

    # --------------------------
    #  Figure 2: Scaling laws
    # --------------------------
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
    fig.suptitle('Scaling Laws Across Pythia Family (WANDA Pruning)', fontsize=13, y=1.03)

    d_ff_arr = np.array([r['d_ff'] for r in good], dtype=float)
    L_arr = np.array([r['L'] for r in good], dtype=float)
    K0_abs_arr = np.array([r['sigmoid_K_0_abs'] for r in good], dtype=float)
    beta_arr = np.array([r['sigmoid_beta'] for r in good])
    g_arr = np.array([r['sigmoid_g_eff'] for r in good])
    params_arr = np.array([r['n_params_M'] for r in good])

    # K₀ (absolute) vs d_ff
    ax = axes[0]
    ax.scatter(d_ff_arr, K0_abs_arr, s=80, c=colors[:len(good)],
               edgecolors='black', lw=0.5, zorder=5)
    for i, r in enumerate(good):
        ax.annotate(r['model'], (d_ff_arr[i], K0_abs_arr[i]),
                    fontsize=7, ha='left', va='bottom')
    ax.set_xlabel('$d_{ff}$ (MLP intermediate dim)', fontsize=11)
    ax.set_ylabel('$K_0$ (neurons)', fontsize=11)
    ax.set_title('Critical Path Count vs MLP Width')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.grid(alpha=0.3, which='both')

    # β vs total parameters
    ax = axes[1]
    ax.scatter(params_arr, beta_arr, s=80, c=colors[:len(good)],
               edgecolors='black', lw=0.5, zorder=5)
    for i, r in enumerate(good):
        ax.annotate(r['model'], (params_arr[i], beta_arr[i]),
                    fontsize=7, ha='left', va='bottom')
    ax.set_xlabel('Parameters (M)', fontsize=11)
    ax.set_ylabel('$\\beta$ (transition steepness)', fontsize=11)
    ax.set_title('Transition Rate vs Model Size')
    ax.set_xscale('log')
    ax.grid(alpha=0.3, which='both')

    # g_eff vs total parameters
    ax = axes[2]
    ax.scatter(params_arr, g_arr, s=80, c=colors[:len(good)],
               edgecolors='black', lw=0.5, zorder=5)
    for i, r in enumerate(good):
        ax.annotate(r['model'], (params_arr[i], g_arr[i]),
                    fontsize=7, ha='left', va='bottom')
    ax.axhline(1.0, color='red', ls=':', lw=1.5, alpha=0.5, label='$g=1$ (strongly coupled)')
    ax.set_xlabel('Parameters (M)', fontsize=11)
    ax.set_ylabel('$g_{eff} = e^{-\\beta}$', fontsize=11)
    ax.set_title('Effective Coupling vs Model Size')
    ax.set_xscale('log')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, which='both')

    plt.tight_layout()
    p = os.path.join(output_dir, 'pythia_scaling_laws.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    paths.append(p); print(f"  Saved: {p}")

    # ---------------------------------------
    #  Figure 3: Summary table as visual
    # ---------------------------------------
    fig, ax = plt.subplots(figsize=(12, max(3, 0.6 * len(good) + 1.5)))
    ax.axis('off')
    ax.set_title('Pythia Family — Sigmoid Pruning Parameters (WANDA)', fontsize=13, pad=20)

    col_labels = ['Model', 'Params', 'L', 'd_ff', 'PPL_base',
                  'K₀ (%)', 'K₀ (abs)', 'β', 'g_eff', 'R²']
    table_data = []
    for r in sorted(good, key=lambda x: x['n_params_M']):
        table_data.append([
            r['model'],
            f'{r["n_params_M"]:.0f}M',
            str(r['L']),
            str(r['d_ff']),
            f'{r["ppl_baseline"]:.1f}',
            f'{r["sigmoid_K_0"]*100:.1f}%',
            str(r['sigmoid_K_0_abs']),
            f'{r["sigmoid_beta"]:.2f}',
            f'{r["sigmoid_g_eff"]:.4f}',
            f'{r["sigmoid_R2"]:.3f}',
        ])

    table = ax.table(cellText=table_data, colLabels=col_labels,
                      loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.6)

    # Style header
    for j in range(len(col_labels)):
        table[0, j].set_facecolor('#4472C4')
        table[0, j].set_text_props(color='white', fontweight='bold')

    # Alternate row colors
    for i in range(1, len(table_data) + 1):
        color = '#D9E2F3' if i % 2 == 0 else 'white'
        for j in range(len(col_labels)):
            table[i, j].set_facecolor(color)

    plt.tight_layout()
    p = os.path.join(output_dir, 'pythia_parameter_heatmap.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    paths.append(p); print(f"  Saved: {p}")

    return paths

### Main

In [ ]:
def main():
    parser = argparse.ArgumentParser(
        description='Pythia family pruning scaling laws')
    parser.add_argument('--models', nargs='*', default=None,
                        help='Model keys (e.g., 70m 410m 1b) or "all"')
    parser.add_argument('--output', default=OUTPUT_DIR)
    parser.add_argument('--eval-tokens', type=int, default=EVAL_MAX_TOKENS)
    parser.add_argument('--n-calib', type=int, default=N_CALIB_SAMPLES)
    args, unknown = parser.parse_known_args()

    eval_max_tokens = args.eval_tokens
    n_calib_samples = args.n_calib

    # Select models
    if args.models is None:
        model_keys = DEFAULT_MODELS
    elif args.models == ['all']:
        model_keys = list(PYTHIA_MODELS.keys())
    else:
        model_keys = [m for m in args.models if m in PYTHIA_MODELS]

    os.makedirs(args.output, exist_ok=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    print("=" * 60)
    print("  PYTHIA FAMILY — EFFECTIVE COUPLING VIA WANDA PRUNING")
    print("=" * 60)
    print(f"  Device: {device}")
    if device == 'cuda':
        print(f"  GPU: {torch.cuda.get_device_name()}")
        # print(f"  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"  Models: {model_keys}")
    print(f"  K fractions: {len(K_FRACTIONS)} levels from "
          f"{K_FRACTIONS[0]:.0%} to {K_FRACTIONS[-1]:.0%}")
    print(f"  Eval tokens: {eval_max_tokens:,}")
    print(f"  Calibration samples: {n_calib_samples}")

    t_total = time.time()
    results = []

    for mk in model_keys:
        try:
            r = run_single_model(mk, device=device,
                                 eval_max_tokens=eval_max_tokens,
                                 n_calib_samples=n_calib_samples)
            results.append(r)
        except Exception as e:
            print(f"\n  ✗ {mk} FAILED: {e}")
            gc.collect()
            if device == 'cuda':
                torch.cuda.empty_cache()

    # Save results
    with open(os.path.join(args.output, 'pythia_results.json'), 'w') as f:
        json.dump(results, f, indent=2)

    # Print summary table
    print(f"\n{'═'*90}")
    print(f"  {'Model':<8} {'Params':>8} {'L':>3} {'d_ff':>6} {'PPL':>8} "
          f"{'K₀(%)':>7} {'K₀(abs)':>8} {'β':>7} {'g_eff':>7} {'R²':>6}")
    print(f"{'─'*90}")
    for r in sorted(results, key=lambda x: x['n_params_M']):
        if r.get('sigmoid_R2') is not None:
            print(f"  {r['model']:<8} {r['n_params_M']:>7.0f}M {r['L']:>3} "
                  f"{r['d_ff']:>6} {r['ppl_baseline']:>8.1f} "
                  f"{r['sigmoid_K_0']*100:>6.1f}% {r['sigmoid_K_0_abs']:>8} "
                  f"{r['sigmoid_beta']:>7.2f} {r['sigmoid_g_eff']:>7.4f} "
                  f"{r['sigmoid_R2']:>6.3f}")
        else:
            print(f"  {r['model']:<8} {r['n_params_M']:>7.0f}M {r['L']:>3} "
                  f"{r['d_ff']:>6} {r['ppl_baseline']:>8.1f}  {'FAILED':>40}")
    print(f"{'═'*90}")

    # Scaling laws
    scaling = fit_scaling_laws(results)
    if scaling:
        with open(os.path.join(args.output, 'pythia_scaling_laws.json'), 'w') as f:
            json.dump(scaling, f, indent=2)

    # Plots
    print(f"\n  Generating plots ...")
    make_plots(results, scaling, args.output)

    dt = time.time() - t_total
    print(f"\n  Total runtime: {dt/60:.1f} min")
    print("  Done!")


if __name__ == '__main__':
    main()

  PYTHIA FAMILY — EFFECTIVE COUPLING VIA WANDA PRUNING
  Device: cuda
  GPU: Tesla T4
  Models: ['14m', '31m', '70m', '160m', '410m', '1b', '1.4b']
  K fractions: 24 levels from 1% to 100%
  Eval tokens: 40,000
  Calibration samples: 128

════════════════════════════════════════════════════════════
  EleutherAI/pythia-14m  (L=6, H=128, d_ff=512)
════════════════════════════════════════════════════════════
  Loading model ...

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/28.1M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

...

  [58s]
  Baseline perplexity ...  11.00  [16s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/c4' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/c4' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

  [6s]
  Computing WANDA scores ...  [0s]
  Sweeping 24 sparsity levels ...
    K= 1.0% (   81/8192)  ppl= 648049.62  recovery=0.1792  [17s]
    K= 2.0% (  163/8192)  ppl= 584423.44  recovery=0.1806  [17s]
    K= 3.0% (  245/8192)  ppl= 303614.82  recovery=0.1899  [16s]
    K= 4.0% (  327/8192)  ppl= 174401.44  recovery=0.1987  [16s]
    K= 5.0% (  409/8192)  ppl= 116942.30  recovery=0.2055  [16s]
    K= 7.5% (  614/8192)  ppl= 101722.89  recovery=0.2080  [17s]
    K=10.0% (  819/8192)  ppl= 137209.46  recovery=0.2027  [16s]
    K=12.5% ( 1024/8192)  ppl= 140944.82  recovery=0.2022  [16s]
    K=15.0% ( 1228/8192)  ppl= 114021.37  recovery=0.2059  [17s]
    K=17.5% ( 1433/8192)  ppl= 101872.04  recovery=0.2079  [17s]
    K=20.0% ( 1638/8192)  ppl=  83966.41  recovery=0.2115  [16s]
    K=20.0% ( 1638/8192)  ppl=  83966.41  recovery=0.2115  [17s]
    K=25.0% ( 2048/8192)  ppl=  52547.15  recovery=0.2206  [17s]
    K=30.0% ( 2457/8192)  ppl=  32276.70  recovery=0.2310  [17s]
    K=35.0% ( 